In [ ]:
# Cell 1: Installation
!pip install yfinance pandas pandas-ta plotly ta mplfinance -q

# Cell 2: Imports
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cell 3: Download Forex Data (EURUSD - 5 years)
print("Downloading EURUSD 5-year data...")
eurusd = yf.download('EURUSD=X', start='2020-01-01', end='2025-01-01', interval='1d')
eurusd.columns = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
eurusd = eurusd[['Open', 'High', 'Low', 'Close', 'Volume']]
eurusd.to_csv('EURUSD_5years.csv')
print(f"Data shape: {eurusd.shape}")
print(eurusd.head())

# Cell 4: Basic Data Exploration
print("="*50)
print("DATA INFO")
print("="*50)
print(eurusd.info())

print("\n" + "="*50)
print("MISSING VALUES")
print("="*50)
print(eurusd.isnull().sum())

print("\n" + "="*50)
print("STATISTICAL SUMMARY")
print("="*50)
print(eurusd.describe())

# Cell 5: Check for Data Quality Issues
# Check for outliers in price (using IQR)
Q1 = eurusd['Close'].quantile(0.25)
Q3 = eurusd['Close'].quantile(0.75)
IQR = Q3 - Q1
outliers = eurusd[(eurusd['Close'] < Q1 - 1.5*IQR) | (eurusd['Close'] > Q3 + 1.5*IQR)]
print(f"Number of outliers in Close price: {len(outliers)}")

# Check for zero or negative prices
invalid_prices = eurusd[eurusd['Close'] <= 0]
print(f"Invalid prices (<=0): {len(invalid_prices)}")

# Cell 6: Visualization - Price Chart
fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=('EURUSD Close Price', 'Daily Returns', 'Volume'),
                    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=eurusd.index, y=eurusd['Close'], 
                         mode='lines', name='Close Price', line=dict(color='blue')), row=1, col=1)

# Daily returns
returns = eurusd['Close'].pct_change() * 100
fig.add_trace(go.Scatter(x=eurusd.index, y=returns, 
                         mode='lines', name='Daily Returns (%)', 
                         line=dict(color='green')), row=2, col=1)

fig.add_trace(go.Bar(x=eurusd.index, y=eurusd['Volume'], 
                     name='Volume', marker_color='orange'), row=3, col=1)

fig.update_layout(height=900, title_text="EURUSD Market Overview", showlegend=True)
fig.show()

# Cell 7: Distribution Analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price distribution
axes[0,0].hist(eurusd['Close'], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Close Price Distribution')
axes[0,0].set_xlabel('Price')
axes[0,0].set_ylabel('Frequency')

# Returns distribution
returns_clean = returns.dropna()
axes[0,1].hist(returns_clean, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0,1].set_title('Daily Returns Distribution')
axes[0,1].set_xlabel('Returns (%)')
axes[0,1].set_ylabel('Frequency')

# Box plot
axes[1,0].boxplot(eurusd['Close'])
axes[1,0].set_title('Box Plot - Close Price')
axes[1,0].set_ylabel('Price')

# QQ plot for normality check
from scipy import stats
stats.probplot(returns_clean, dist="norm", plot=axes[1,1])
axes[1,1].set_title('QQ Plot - Returns Normality')

plt.tight_layout()
plt.show()

# Cell 8: Correlation Analysis
correlation_matrix = eurusd[['Open', 'High', 'Low', 'Close', 'Volume']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.3f')
plt.title('Correlation Matrix - EURUSD')
plt.show()

print("\nKey Correlations:")
print(f"Open-Close correlation: {correlation_matrix.loc['Open','Close']:.4f}")
print(f"High-Low correlation: {correlation_matrix.loc['High','Low']:.4f}")

# Cell 9: Seasonality and Pattern Analysis
# Add time-based features
eurusd['Year'] = eurusd.index.year
eurusd['Month'] = eurusd.index.month
eurusd['Day'] = eurusd.index.day
eurusd['DayOfWeek'] = eurusd.index.dayofweek

# Monthly average prices
monthly_avg = eurusd.groupby('Month')['Close'].mean()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
monthly_avg.plot(kind='bar', color='skyblue')
plt.title('Average Close Price by Month')
plt.xlabel('Month')
plt.ylabel('Average Price')
plt.xticks(rotation=0)

# Day of week patterns
weekly_avg = eurusd.groupby('DayOfWeek')['Close'].mean()
weekly_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
plt.subplot(1, 2, 2)
weekly_avg.plot(kind='bar', color='lightcoral')
plt.title('Average Close Price by Day')
plt.xlabel('Day of Week')
plt.ylabel('Average Price')
plt.xticks(range(5), weekly_labels, rotation=45)

plt.tight_layout()
plt.show()

# Cell 10: Volatility Analysis
eurusd['Returns'] = eurusd['Close'].pct_change()
eurusd['Volatility'] = eurusd['Returns'].rolling(window=20).std() * np.sqrt(252)  # Annualized

plt.figure(figsize=(14, 6))
plt.plot(eurusd.index, eurusd['Volatility'] * 100, color='purple')
plt.title('EURUSD Rolling Volatility (20-day window, annualized)')
plt.xlabel('Date')
plt.ylabel('Volatility (%)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Average volatility: {eurusd['Volatility'].mean()*100:.2f}%")
print(f"Max volatility: {eurusd['Volatility'].max()*100:.2f}%")
print(f"Min volatility: {eurusd['Volatility'].min()*100:.2f}%")

# Cell 11: Save Clean Data
eurusd_clean = eurusd[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
eurusd_clean.to_csv('EURUSD_clean.csv')
print(f"Clean data saved. Shape: {eurusd_clean.shape}")